# Market Sentiment — All-in-One

All project code in one notebook. Run cells in order.

In [1]:
import json
import re
import sys
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    GenerationConfig,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore", message=".*use_reentrant.*", category=UserWarning)
import transformers.modeling_utils  # noqa: F401

# --- Config ---
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_ID = "cyrilzhang/financial_phrasebank_split"
LABELS = ["Greed", "Neutral", "Panic"]
LABEL_MAP = {
    "positive": "Greed",
    "neutral": "Neutral",
    "negative": "Panic",
}
LABEL_MAP_INT = {0: "Panic", 1: "Neutral", 2: "Greed"}
INSTRUCTION_TEMPLATE = """Instruction:
Classify the market sentiment as Greed, Panic, or Neutral.

Input:
{input_text}

Output:
{label}"""


@dataclass
class LoRAConfig:
    name: str
    learning_rate: float
    lora_r: int
    lora_alpha: int = 16
    epochs: int = 3
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    max_seq_length: int = 256
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    logging_steps: int = 10
    save_strategy: str = "epoch"
    bf16: bool = True
    fp16: bool = False
    max_steps: Optional[int] = 100
    lr_scheduler_type: str = "cosine"


def get_experiment_configs() -> List[LoRAConfig]:
    return [
        LoRAConfig(name="config_1", learning_rate=2e-4, lora_r=8, max_steps=100),
        LoRAConfig(name="config_2", learning_rate=1e-4, lora_r=8, max_steps=100),
        LoRAConfig(name="config_3", learning_rate=2e-4, lora_r=16, max_steps=100),
        LoRAConfig(name="config_4", learning_rate=1e-4, lora_r=16, max_steps=100),
        LoRAConfig(name="config_5", learning_rate=5e-5, lora_r=16, max_steps=100),
    ]



print("Imports and config OK.")

Imports and config OK.


In [2]:
# --- Data ---
def map_labels(example: dict) -> dict:
    lab = example["label"]
    if isinstance(lab, int):
        example["sentiment"] = LABEL_MAP_INT.get(lab, "Neutral")
    else:
        key = lab.lower() if isinstance(lab, str) else lab
        example["sentiment"] = LABEL_MAP.get(key, "Neutral")
    return example


def format_instruction(example: dict) -> dict:
    example["text"] = INSTRUCTION_TEMPLATE.format(
        input_text=example["sentence"].strip(),
        label=example["sentiment"],
    )
    example["target_label"] = example["sentiment"]
    return example


def prepare_splits(
    eval_size: int = 100,
    seed: int = 42,
) -> Tuple[Dataset, Dataset]:
    raw = load_dataset(DATASET_ID, split="train")
    ds = raw.map(map_labels, desc="Mapping labels")
    ds = ds.map(format_instruction, desc="Formatting")
    shuffled = ds.shuffle(seed=seed)
    n = len(shuffled)
    eval_size = min(eval_size, max(1, n - 1))
    train_ds = shuffled.select(range(n - eval_size))
    eval_ds = shuffled.select(range(n - eval_size, n))
    return train_ds, eval_ds


print("1. Data...")
OUTPUT_DIR = Path("outputs")
EVAL_SIZE = 100
CONFIG_NAME = None  # or "config_1" for single run
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train_ds, eval_ds = prepare_splits(eval_size=EVAL_SIZE)
print(f"   train={len(train_ds)}, eval={len(eval_ds)}")

1. Data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/578 [00:00<?, ?B/s]

data/train-00000-of-00001-500b955d77e781(…):   0%|          | 0.00/376k [00:00<?, ?B/s]

data/test-00000-of-00001-08257c1f0eb1877(…):   0%|          | 0.00/42.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4361 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/485 [00:00<?, ? examples/s]

Mapping labels:   0%|          | 0/4361 [00:00<?, ? examples/s]

Formatting:   0%|          | 0/4361 [00:00<?, ? examples/s]

   train=4261, eval=100


In [3]:
# --- Model (load base + LoRA) ---
def _bnb_available() -> bool:
    try:
        import bitsandbytes
        ver = getattr(bitsandbytes, "__version__", "0")
        return tuple(int(x) for x in ver.split(".")[:3]) >= (0, 46, 1)
    except (ImportError, ValueError):
        return False


def get_tokenizer(model_id: str = MODEL_ID):
    t = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if t.pad_token is None:
        t.pad_token = t.eos_token
    return t


def get_base_model_and_tokenizer(
    model_id: str = MODEL_ID,
    use_4bit: bool = True,
    device_map: str = "auto",
):
    tokenizer = get_tokenizer(model_id)
    if use_4bit and not _bnb_available():
        use_4bit = False
    model_kwargs = {
        "trust_remote_code": True,
        "device_map": device_map,
        "dtype": torch.bfloat16 if not use_4bit else None,
    }
    if use_4bit:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    if use_4bit:
        model = prepare_model_for_kbit_training(model)
    return model, tokenizer


def apply_lora(
    model,
    r: int = 8,
    lora_alpha: int = 16,
    target_modules=None,
):
    if target_modules is None:
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ]
    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    return get_peft_model(model, lora_config)

print("Model helpers OK.")

Model helpers OK.


In [4]:
# --- Train ---
def tokenize_for_training(
    dataset: Dataset,
    tokenizer,
    max_length: int = 256,
    text_column: str = "text",
):
    def tokenize(examples):
        out = tokenizer(
            examples[text_column],
            truncation=True,
            max_length=max_length,
            padding="max_length",
            return_tensors=None,
        )
        out["labels"] = [ids.copy() for ids in out["input_ids"]]
        return out
    return dataset.map(
        tokenize,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing",
    )


def train(
    train_dataset: Dataset,
    eval_dataset: Optional[Dataset],
    lora_config: LoRAConfig,
    output_dir: str = "outputs",
    use_4bit: bool = True,
) -> str:
    model, tokenizer = get_base_model_and_tokenizer(MODEL_ID, use_4bit=use_4bit)
    model = apply_lora(
        model,
        r=lora_config.lora_r,
        lora_alpha=lora_config.lora_alpha,
    )
    run_dir = Path(output_dir) / lora_config.name
    run_dir.mkdir(parents=True, exist_ok=True)
    train_tok = tokenize_for_training(
        train_dataset,
        tokenizer,
        max_length=lora_config.max_seq_length,
    )
    eval_tok = None
    if eval_dataset and len(eval_dataset) > 0:
        eval_tok = tokenize_for_training(
            eval_dataset,
            tokenizer,
            max_length=lora_config.max_seq_length,
        )
    batch_size = (
        lora_config.per_device_train_batch_size
        * lora_config.gradient_accumulation_steps
    )
    use_max_steps = getattr(lora_config, "max_steps", None) is not None
    if use_max_steps:
        total_steps = lora_config.max_steps
        warmup_steps = max(1, int(total_steps * lora_config.warmup_ratio))
        num_train_epochs = 1
    else:
        total_steps = (len(train_tok) // batch_size) * lora_config.epochs
        warmup_steps = max(1, int(total_steps * lora_config.warmup_ratio))
        num_train_epochs = lora_config.epochs
    args = TrainingArguments(
        output_dir=str(run_dir),
        max_steps=total_steps if use_max_steps else -1,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=lora_config.per_device_train_batch_size,
        gradient_accumulation_steps=lora_config.gradient_accumulation_steps,
        learning_rate=lora_config.learning_rate,
        weight_decay=lora_config.weight_decay,
        warmup_steps=warmup_steps,
        lr_scheduler_type=getattr(lora_config, "lr_scheduler_type", "cosine"),
        logging_steps=lora_config.logging_steps,
        save_strategy="no" if use_max_steps else lora_config.save_strategy,
        bf16=lora_config.bf16,
        fp16=lora_config.fp16,
        remove_unused_columns=False,
        report_to="none",
    )
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=eval_tok,
        data_collator=collator,
    )
    trainer.train()
    trainer.save_model(str(run_dir / "final"))
    tokenizer.save_pretrained(str(run_dir / "final"))
    return str(run_dir / "final")

print("Train helpers OK.")

Train helpers OK.


In [5]:
# --- Eval ---
def extract_predicted_label(completion: str) -> str:
    completion = (completion or "").strip()
    for label in LABELS:
        if label.lower() in completion.lower():
            if re.search(rf"\b{re.escape(label)}\s*$", completion, re.IGNORECASE):
                return label
            if label in completion:
                return label
    for label in LABELS:
        if label.lower() in completion.lower():
            return label
    return LABELS[1]


def run_inference(
    model,
    tokenizer,
    sentences: List[str],
    batch_size: int = 8,
    seed: int = 42,
) -> List[str]:
    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    gen_config = GenerationConfig(
        max_new_tokens=16,
        do_sample=True,
        pad_token_id=pad_id,
        temperature=0.3,
        top_p=0.9,
        top_k=50,
    )
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    prefix = "Classify the market sentiment as Greed, Panic, or Neutral.\n\nInput:\n"
    predictions = []
    for i in tqdm(range(0, len(sentences), batch_size), desc="Eval"):
        batch = sentences[i : i + batch_size]
        prompts = [f"Instruction:\n{prefix}{s}\n\nOutput:\n" for s in batch]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, generation_config=gen_config)
        for j, gen in enumerate(out):
            prompt_len = inputs["attention_mask"][j].sum().item()
            gen_text = tokenizer.decode(gen[prompt_len:], skip_special_tokens=True)
            predictions.append(extract_predicted_label(gen_text))
    tokenizer.padding_side = old_side
    return predictions


def evaluate_model(
    model,
    tokenizer,
    eval_dataset: Dataset,
    sentence_key: str = "sentence",
    label_key: str = "target_label",
) -> Tuple[float, float]:
    sentences = [ex[sentence_key] for ex in eval_dataset]
    gold = [ex[label_key] for ex in eval_dataset]
    preds = run_inference(model, tokenizer, sentences)
    return accuracy_score(gold, preds), f1_score(gold, preds, average="macro", zero_division=0)


def evaluate_base_model(eval_dataset: Dataset) -> Tuple[float, float]:
    model, tokenizer = get_base_model_and_tokenizer(MODEL_ID, use_4bit=False)
    model.eval()
    return evaluate_model(model, tokenizer, eval_dataset)


def evaluate_finetuned_model(
    checkpoint_dir: str,
    eval_dataset: Dataset,
) -> Tuple[float, float]:
    ckpt = Path(checkpoint_dir)
    tokenizer_path = str(ckpt) if (ckpt / "tokenizer_config.json").exists() else MODEL_ID
    tokenizer = get_tokenizer(tokenizer_path)
    base_model, _ = get_base_model_and_tokenizer(MODEL_ID, use_4bit=False)
    model = PeftModel.from_pretrained(base_model, str(ckpt))
    model.eval()
    return evaluate_model(model, tokenizer, eval_dataset)

print("Eval helpers OK.")

Eval helpers OK.


In [6]:
print("2. Base model eval...")
base_acc, base_f1 = evaluate_base_model(eval_ds)
print(f"   acc={base_acc:.2%}, f1={base_f1:.4f}")

2. Base model eval...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Eval: 100%|██████████| 13/13 [00:13<00:00,  1.01s/it]

   acc=40.00%, f1=0.3055


In [7]:
print("3. Train LoRA...")
configs = get_experiment_configs()
if CONFIG_NAME:
    configs = [c for c in configs if c.name == CONFIG_NAME]
    if not configs:
        raise ValueError(f"Unknown config: {CONFIG_NAME}")
for c in configs:
    print(f"   {c.name}...")
    train(train_ds, eval_ds, c, output_dir=str(OUTPUT_DIR), use_4bit=True)
    print(f"   {c.name} done.")
print("   All training done.")

3. Train LoRA...
   config_1...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing:   0%|          | 0/4261 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss
10,3.432766
20,2.186439
30,1.958078
40,1.877239
50,1.915926
60,1.937547
70,1.827078
80,1.806362
90,1.887681
100,1.789353


...

Step,Training Loss
10,3.657713
20,2.539672
30,2.016386
40,1.938963
50,1.967658
60,1.998453
70,1.873330
80,1.854245
90,1.939931
100,1.856547


   config_4 done.
   config_5...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
10,3.799435
20,3.080915
30,2.243409
40,2.029159
50,2.029706
60,2.061342
70,1.929214
80,1.915191
90,1.996812
100,1.928286


   config_5 done.
   All training done.


## 3b. Hyperparameter tuning summary

Configs trained (fixed 100 steps): learning rate and LoRA rank varied.

In [8]:
print("Hyperparameter tuning (configs just trained):")
for c in configs:
    print(f"   {c.name}: lr={c.learning_rate}, lora_r={c.lora_r}, max_steps={c.max_steps}")

Hyperparameter tuning (configs just trained):
   config_1: lr=0.0002, lora_r=8, max_steps=100
   config_2: lr=0.0001, lora_r=8, max_steps=100
   config_3: lr=0.0002, lora_r=16, max_steps=100
   config_4: lr=0.0001, lora_r=16, max_steps=100
   config_5: lr=5e-05, lora_r=16, max_steps=100


In [9]:
print("4. Eval fine-tuned...")
results = {"base_model": {"accuracy": base_acc, "f1_macro": base_f1}, "configs": {}}
for c in get_experiment_configs():
    ckpt = OUTPUT_DIR / c.name / "final"
    if not ckpt.exists():
        continue
    print(f"   {c.name}...")
    acc, f1 = evaluate_finetuned_model(str(ckpt), eval_ds)
    results["configs"][c.name] = {"accuracy": acc, "f1_macro": f1}
    print(f"   {c.name} acc={acc:.2%}, f1={f1:.4f}")
if results["configs"]:
    best_name = max(results["configs"].keys(), key=lambda k: results["configs"][k]["f1_macro"])
    best = results["configs"][best_name]
    print(f"   Best config: {best_name} (acc={best['accuracy']:.2%}, f1={best['f1_macro']:.4f})")
print("   Eval done.")

4. Eval fine-tuned...
   config_1...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Eval: 100%|██████████| 13/13 [00:18<00:00,  1.41s/it]


   config_1 acc=58.00%, f1=0.5917
   config_2...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Eval: 100%|██████████| 13/13 [00:17<00:00,  1.37s/it]


   config_4 acc=56.00%, f1=0.5676
   config_5...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Eval: 100%|██████████| 13/13 [00:17<00:00,  1.37s/it]

   config_5 acc=47.00%, f1=0.3595
   Best config: config_3 (acc=62.00%, f1=0.6233)
   Eval done.


In [10]:
print("5. Summary")
rows = [{"model": "Base", **results["base_model"]}]
for name, m in results["configs"].items():
    rows.append({"model": name, **m})
for r in rows:
    print(f"   {r['model']:12} acc={r['accuracy']:.2%}  f1={r['f1_macro']:.4f}")
if results["configs"]:
    best_name = max(results["configs"].keys(), key=lambda k: results["configs"][k]["f1_macro"])
    best = results["configs"][best_name]
    results["best_config"] = best_name
    results["best_config_metrics"] = best
    print(f"   --> Best config: {best_name} (acc={best['accuracy']:.2%}, f1={best['f1_macro']:.4f})")
out_path = OUTPUT_DIR / "results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"   Saved {out_path}")

5. Summary
   Base         acc=40.00%  f1=0.3055
   config_1     acc=58.00%  f1=0.5917
   config_2     acc=58.00%  f1=0.5908
   config_3     acc=62.00%  f1=0.6233
   config_4     acc=56.00%  f1=0.5676
   config_5     acc=47.00%  f1=0.3595
   --> Best config: config_3 (acc=62.00%, f1=0.6233)
   Saved outputs/results.json
